In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchinfo import summary
from torch import nn    
from torch.nn import functional as F
from torch.utils.data import random_split
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Dataset

In [ ]:
DATA_DIR = "../datasets/waveforms"

In [ ]:
class WavefromDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.mixture_files = sorted([f for f in os.listdir(directory) if f.endswith('_mixture.npy')])

    def __len__(self):
        return len(self.mixture_files)

    def __getitem__(self, idx):

        mixture_path = os.path.join(self.directory, self.mixture_files[idx])
        vocal_path = mixture_path.replace('_mixture.npy', '_vocals.npy')
        instrumental_path = mixture_path.replace('_mixture.npy', '_instrumental.npy')

        mixture = np.load(mixture_path)
        vocals = np.load(vocal_path)
        instrumental = np.load(instrumental_path)

        X = torch.from_numpy(mixture).float().unsqueeze(0)
        Y = torch.from_numpy(np.stack([vocals, instrumental], axis=0)).float()
        
        pad = (-X.shape[-1]) % 16
        X = F.pad(X, (0, pad))
        Y = F.pad(Y, (0, pad))
        return X, Y

In [ ]:
train_dataset = WavefromDataset(os.path.join(DATA_DIR,"train"))
val_dataset = WavefromDataset(os.path.join(DATA_DIR,"val"))

print(len(train_dataset))
print(len(val_dataset))

6896
981


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=20,
    shuffle=False
)

Model

In [ ]:
class Demucs(nn.Module):
    def __init__(self,inchannels = 1, outchannels = 2, features = [32, 64, 128, 256]):
        super().__init__()

        self.features = features
        self.inchannels = inchannels
        self.outchannels = outchannels

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        in_features = inchannels
        for feature in self.features:
            en = self.Encoder(in_features,feature)
            self.encoders.append(en)

            de = self.Decoder(2*feature, in_features)
            self.decoders.append(de)

            in_features = feature

        self.decoders = self.decoders[::-1]

        self.bottleneckLSTM = nn.LSTM(
                input_size=self.features[-1],
                hidden_size=self.features[-1] // 2,
                num_layers=2,
                batch_first=True,
                bidirectional=True)

        self.final = nn.Conv1d(inchannels,outchannels,kernel_size=1)

        self.sigmoid = nn.Sigmoid()

    def Encoder(self,in_channels,out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels,in_channels,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels,2*in_channels,kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(in_channels,out_channels,kernel_size=8,stride=4,padding=1),
            nn.ReLU()
        )

    def Decoder(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels, 2*in_channels, kernel_size=1),
            nn.GLU(dim=1),
            nn.ConvTranspose1d(in_channels, out_channels,kernel_size=8,stride=4,padding=1),
            nn.ReLU()
        )

    def forward(self,mixture):

        x = mixture
        skip = []

        for en in self.encoders:
            x = en(x)
            skip.append(x)

        x = x.transpose(1, 2)
        x, _ = self.bottleneckLSTM(x)
        x = x.transpose(1, 2)

        skip = skip[::-1]

        for i, de in enumerate(self.decoders):
            diff = skip[i].shape[-1] - x.shape[-1]

            if diff > 0:
                x = F.pad(x, (0, diff))
            elif diff < 0:
                x = F.pad(x, (0, -diff))
                
            x = torch.cat((x, skip[i]), dim=1)
            x = de(x)

        diff = mixture.shape[-1] - x.shape[-1]

        if diff > 0:
            x = F.pad(x, (0, diff))
        elif diff < 0:
            x = x[:, :mixture.shape[-1]]

        return self.final(x)

In [ ]:
model = Demucs(inchannels=1, outchannels=2).to(device)
summary(model, input_size=(20, 1 , 205008), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
Demucs                                   [20, 2, 205008]           --
├─ModuleList: 1-1                        --                        --
│    └─Sequential: 2-1                   [20, 32, 51251]           --
│    │    └─Conv1d: 3-1                  [20, 1, 205008]           4
│    │    └─ReLU: 3-2                    [20, 1, 205008]           --
│    │    └─Conv1d: 3-3                  [20, 2, 205008]           4
│    │    └─GLU: 3-4                     [20, 1, 205008]           --
│    │    └─Conv1d: 3-5                  [20, 32, 51251]           288
│    │    └─ReLU: 3-6                    [20, 32, 51251]           --
│    └─Sequential: 2-2                   [20, 64, 12812]           --
│    │    └─Conv1d: 3-7                  [20, 32, 51251]           3,104
│    │    └─ReLU: 3-8                    [20, 32, 51251]           --
│    │    └─Conv1d: 3-9                  [20, 64, 51251]           2,112
│    │    

In [ ]:
epochs = 50
patience = 6
best_loss_value = float("inf")
lastbest = -1

loss_fn = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.75,
    patience=3,
    min_lr=1e-5
)

train_loss_epochwise = []
val_loss_epochwise = []

MODEL_DIR = "../models/"
PLOT_DIR = "../plots/"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

model_name = "demucs_vocal_inst_sep"

model = model.to(device)

In [ ]:
def save_loss_plots():

    plt.plot(train_loss_epochwise)
    plt.title("Epochwise Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(
        os.path.join(PLOT_DIR, f"{model_name}_train_loss_epochwise.png")
    )
    plt.close()

    plt.plot(val_loss_epochwise)
    plt.title("Epochwise Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(
        os.path.join(PLOT_DIR, f"{model_name}_val_loss_epochwise.png")
    )
    plt.close()

In [ ]:
for epoch in range(epochs):

    model.train()
    running_train_loss = 0.0

    for batch_idx, (X, Y) in enumerate(train_loader):

        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad(set_to_none=True)

        prediction = model(X)

        loss = loss_fn(prediction, Y)

        loss.backward()

        # Prevent excessively large gradients in the LSTM
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0)

        optimizer.step()

        running_train_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] " f"Step [{batch_idx+1}/{len(train_loader)}] " f"Train Loss: {loss.item():.6f}")

    avg_train_loss = running_train_loss / len(train_loader)
    train_loss_epochwise.append(avg_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        for X, Y in val_loader:

            X = X.to(device)
            Y = Y.to(device)

            prediction = model(X)

            loss = loss_fn(prediction, Y)

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_loss_epochwise.append(avg_val_loss)

    print(f"\nEpoch [{epoch+1}/{epochs}] " f"Train Loss: {avg_train_loss:.6f} | "f"Val Loss: {avg_val_loss:.6f} | "f"LR: {optimizer.param_groups[0]['lr']:.7f}")

    save_loss_plots()

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_loss_value:

        best_loss_value = avg_val_loss
        lastbest = epoch

        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_best.pt"))
        print(f"New best model saved: {best_loss_value:.6f}")

    elif epoch - lastbest >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

torch.save(
    model.state_dict(),
    os.path.join(MODEL_DIR, f"{model_name}_last.pt")
)


Epoch [1/50] Step [1/345] Train Loss: 0.751353


KeyboardInterrupt: 